# Overture Maps Places Downloader

Downloads place data from Overture Maps (S3) for a defined region, grouped by category, and saves each category as a separate Parquet file.

**Output:** one `.parquet` file per category in the configured output directory.

In [ ]:
import subprocess, sys

for pkg in ['duckdb', 'pandas', 'pyarrow']:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"✓ {pkg} installed")

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# ── Configuration ──────────────────────────────────────────────────────────────

OVERTURE_RELEASE = "2026-01-21.0"

# Bounding box for the region of interest — Australia
BBOX = {
    "min_lon": 113.0,
    "min_lat": -44.0,
    "max_lon": 154.0,
    "max_lat": -10.0,
}

# ── Categories to download ─────────────────────────────────────────────────────
# Each entry is a value for `categories.primary` in Overture Maps.
# Edit this list to control what gets downloaded.
#
# Full Overture Maps place taxonomy (https://docs.overturemaps.org/schema/reference/places/place/):
#
# Active life / fitness:
#   fitness_center, gym, yoga_studio, pilates_studio, crossfit, dance_studio,
#   martial_arts, swimming_pool, sports_club, rock_climbing, batting_cages,
#   golf_course, tennis_court, ski_resort, skate_park, bowling_alley,
#   amusement_park, miniature_golf, go_kart, laser_tag, trampoline_park
#
# Food & drink:
#   restaurant, cafe, fast_food_restaurant, bar, bakery, ice_cream, food_court,
#   food_truck, brewery, winery, cocktail_bar, sports_bar, buffet, pub,
#   juice_bar, tea_house, dessert_shop, pizza_restaurant, sushi_restaurant
#
# Health & medical:
#   hospital, medical_clinic, pharmacy, dentist, physiotherapist, chiropractor,
#   optometrist, veterinarian, urgent_care, dialysis_center, blood_bank,
#   mental_health_clinic, acupuncture, massage_therapy, nursing_home
#
# Shopping & retail:
#   supermarket, convenience_store, shopping_mall, clothing_store, electronics_store,
#   hardware_store, furniture_store, bookstore, sports_store, toy_store,
#   jewelry_store, department_store, discount_store, florist, pet_store,
#   liquor_store, gift_shop, art_gallery, antique_shop, thrift_store
#
# Education:
#   school, university, college, kindergarten, tutoring_center, library,
#   driving_school, language_school, vocational_school, museum
#
# Accommodation & travel:
#   hotel, motel, hostel, bed_and_breakfast, resort, campground, vacation_rental,
#   airport, train_station, bus_station, car_rental, taxi, ferry_terminal
#
# Finance:
#   bank, atm, insurance, financial_advisor, currency_exchange, credit_union
#
# Services:
#   fuel_station, car_wash, auto_repair, hair_salon, nail_salon, laundromat,
#   postal_service, real_estate, legal_services, accounting, advertising,
#   security, cleaning_service, photographer, event_venue, funeral_home
#
# Public & infrastructure:
#   park, playground, public_toilet, police, fire_station, courthouse,
#   city_hall, embassy, post_office, community_center, place_of_worship,
#   cemetery, recycling_center, parking, electric_vehicle_charging_station
#
# Entertainment:
#   movie_theater, theater, concert_hall, casino, nightclub, karaoke,
#   comedy_club, escape_room, arcade, zoo, aquarium, botanical_garden,
#   theme_park, water_park, stadium, arena

CATEGORIES = [
    "fitness_center",
    "gym",
    "yoga_studio",
    "pilates_studio",
    "crossfit",
    "swimming_pool",
    "sports_club",
    "dance_studio",
    "martial_arts",
]

# Output directory
OUTPUT_DIR = Path("/opt/workspace/heenco")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Overture release : {OVERTURE_RELEASE}")
print(f"Bounding box     : lon {BBOX['min_lon']}–{BBOX['max_lon']},  lat {BBOX['min_lat']}–{BBOX['max_lat']}")
print(f"Categories       : {len(CATEGORIES)}")
print(f"Output directory : {OUTPUT_DIR.resolve()}")

In [ ]:
def download_category(con: duckdb.DuckDBPyConnection, category: str, bbox: dict, release: str) -> pd.DataFrame:
    """Query Overture Maps S3 for a single primary category within the bounding box.
    Returns all available columns; geometry is stored as both lon/lat and WKT.
    """
    s3_path = f"s3://overturemaps-us-west-2/release/{release}/theme=places/type=place/**/*.parquet"
    query = f"""
    SELECT
        * EXCLUDE (geometry),
        ST_X(geometry)        AS longitude,
        ST_Y(geometry)        AS latitude,
        ST_AsText(geometry)   AS geometry_wkt
    FROM read_parquet('{s3_path}')
    WHERE categories.primary = '{category}'
      AND bbox.xmin < {bbox['max_lon']}
      AND bbox.xmax > {bbox['min_lon']}
      AND bbox.ymin < {bbox['max_lat']}
      AND bbox.ymax > {bbox['min_lat']}
    """
    return con.execute(query).df()


# Initialise DuckDB with required extensions
con = duckdb.connect()
con.execute("INSTALL httpfs;  LOAD httpfs;")
con.execute("INSTALL spatial; LOAD spatial;")
print("✓ DuckDB ready")

In [ ]:
results = {}   # category → row count

for category in CATEGORIES:
    out_path = OUTPUT_DIR / f"{category}.parquet"

    if out_path.exists():
        existing = pd.read_parquet(out_path)
        print(f"  skip  {category:30s} ({len(existing):>6,} rows) — already downloaded")
        results[category] = len(existing)
        continue

    print(f"  fetch {category:30s} …", end=" ", flush=True)
    try:
        df = download_category(con, category, BBOX, OVERTURE_RELEASE)
        df.to_parquet(out_path, index=False)
        print(f"{len(df):>6,} rows  →  {out_path.name}")
        results[category] = len(df)
    except Exception as exc:
        print(f"ERROR: {exc}")
        results[category] = -1

con.close()
print("\n✓ Done")

In [ ]:
# Summary table
print(f"{'Category':<30} {'Rows':>8}  {'File'}")
print("-" * 60)
total = 0
for cat, n in results.items():
    fname = f"{cat}.parquet" if n >= 0 else "—"
    status = f"{n:>8,}" if n >= 0 else "   ERROR"
    print(f"{cat:<30} {status}  {fname}")
    if n > 0:
        total += n
print("-" * 60)
print(f"{'TOTAL':<30} {total:>8,}")
print(f"\nFiles saved to: {OUTPUT_DIR.resolve()}")